# 2. Forecast scenarios

Unit: America's Debt Crisis. Concept level: **understand → analyze → challenge → capstone**.

Data: the course dataset lives in one deterministic DuckDB file, `../../data/analytics.duckdb`, seeded
from `seed/seed.sql` (identical inside the Docker CLI). Every number here is
stable: rerun any notebook years from now and it reproduces the same answers,
because the seed uses fixed anchors (the course video's figures) plus
deterministic noise -- never the random number generator.

Workflow: run cells top to bottom. A `# TASK` comment marks a cell you should
edit; answer in the markdown cell just below when asked.

The cinematic device of the video: **let the bond market speak.** The CBO
can forecast Congress's numbers, but the market prices them continuously --
the 2026 30-year auction at 5.3% was the clearest signal in the dataset.

Three levels of thinking:
1. **Baseline** -- debt service vs growth, interest flat at 2.5%, growth heals ~1.2% -> 4.1%.
2. **Static sensitivity** -- what if the market reprices the 30-year (higher effective r)?
3. **The crossover clock** -- the first year growth stays above interest. The single most quotable number.

### 2.1 The model in ten lines

Growth g(t) rises linearly from `g2026` (+`GROWTH_STEP`/yr) up to the
`RATE_G` ceiling; effective interest `r` is fixed at `BASELINE_R` unless you
overrule it. The share `interest_pct_gdp` drifts with `g - r`. All
deterministic -- see `../../scripts/model.py`. Baseline crossover: **2031**.

In [ ]:
# Bootstrap
%matplotlib inline
import sys
from pathlib import Path
import duckdb

sys.path.insert(0, "../../scripts")   # the g-vs-r model used in 02-03
import model as m

DB = "../../data/analytics.duckdb"
con = duckdb.connect(DB)

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({"figure.figsize": (8, 4.4),
                     "axes.grid": True, "grid.alpha": 0.35,
                     "font.size": 10})

In [ ]:
base = m.project(30)                       # 2026 -> 2055
crossover = m.crossover(base)
print(f"BASELINE_CROSSOVER_YEAR = {crossover}")
assert crossover == 2031, "the video's ~2031 breathing-room anchor"

base_df = pd.DataFrame(base)
fig, ax = plt.subplots()
ax.plot(base_df.year, base_df.g, label="nominal growth g (%)", color="C0")
ax.plot(base_df.year, base_df.r, label="effective interest r (%)", color="C3")
ax.axvline(crossover, color="C2", ls=":", lw=1)
ax.text(crossover, 1.0, "crossover", color="C2", ha="center")
ax.legend(); ax.set(xlabel="year", ylabel="%/year",
                    title="Baseline: g vs r (debt breathes once g >= r)")
plt.show()

### 2.2 The scenario table (flipping the market's rate)

`model.sensitivity()` raises effective r by each stress and reports the new
crossover year.

In [ ]:
sens = m.sensitivity()
print(pd.DataFrame(sens).to_string(index=False))
for row in sens:
    print(f"r +{row['r_bp']}bp -> crossover {row['crossover']}")
assert sens[0]["crossover"] > 2031 and sens[-1]["crossover"] == 2038

In [ ]:
fig, ax = plt.subplots()
for row in sens:
    rows = m.project(30, r=round(m.BASELINE_R + row["r_bp"] / 100, 3))
    df = pd.DataFrame(rows)
    x = row["crossover"]
    label = f"r = {df.r.iloc[0]}% (cross {x})" if x else f"r = {df.r.iloc[0]}% (never)"
    ax.plot(df.year, df.g_minus_r, label=label)
ax.axhline(0, color="C2", ls="--")
ax.set(xlabel="year", ylabel="g - r (pp)", title="g - r staying below zero -> no breathing room")
ax.legend(fontsize=8)
plt.show()

### 2.3 The interest decade

The `interest_outlook` table carries a 10-year net-interest sum of
**$20,235B ~ $20.2T**, comfortably above the video's "$16T" headline. The gap
is exactly the tension of the lecture: the 2.5% effective-r framework sits far
below the market's own 5.3%.

In [ ]:
out = con.execute("SELECT * FROM interest_outlook ORDER BY fiscal_year").fetchdf()
decade_total = out.net_interest_billions.sum()
print(f"10-year net interest (2026-2036) = ${decade_total:,.0f}B ~ ${decade_total/1000:.1f}T")
assert decade_total > 16_000
fig, ax = plt.subplots()
ax.bar(out.fiscal_year.astype(str), out.net_interest_billions)
ax.set(xlabel="fiscal year", ylabel="$B", title="Growing net-interest bill")
plt.show()

### 2.4 Checkpoint
1. Under the **baseline**, which year does growth first keep up with interest?
2. Raising r by **84bp** pushes that year to ___; by 158bp to ___ (printed above).
3. A 30-year sell-off works through which channel, **g** or **r**?
4. **Reft**: how slow could the *growth healing* be (`GROWTH_STEP`) so `g2026 = 1.2`
   never crosses 2.5% inside `project(30)`?  Try it below -- that's the pessimistic case.

In [ ]:
# TASK: shrink GROWTH_STEP until the crossover disappears in the window.
for step in (0.2, 0.15, 0.1, 0.05):
    m.GROWTH_STEP = step
    m.GROWTH_2026 = 1.2
    print(f"GROWTH_STEP={step}  crossover={m.crossover(m.project(30))}")
m.GROWTH_STEP = 0.26   # reset everything for later notebooks
m.GROWTH_2026 = 1.2
print("reset OK")

---
End of notebook 2. Next: `03-growth-plan-challenge` -- stress the Bessent "333" plan.